# SPOTnet — Step 06: Iterative Inpainting

This notebook applies **iterative specular removal** using the trained SPOTnet detector and LaMa inpainting.

**Pipeline:**
```
Input Frame → SPOTnet Detection → Mask Dilation → LaMa Inpainting → Repeat until clean
```

Based on EndoSRR (Tobyzai et al.), we iterate until the specular ratio drops below a threshold.

**Requirements:**
```
pip install simple-lama-inpainting
```

## 1. Imports & Setup

In [ ]:
import os, glob, math
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
from simple_lama_inpainting import SimpleLama

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# Paths — update to your dataset
FRAMES_DIR  = 'frames'
OUTPUT_DIR  = 'inpainted_results'
CKPT_PATH   = 'runs/detector_spotnet/best.pt'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Inference settings
TILE     = 384
OVERLAP  = 96
BASE_CH  = 32
BIN_THR  = 0.5
DIL_K    = 15    # dilation kernel size
MAX_ITER = 5     # max inpainting iterations
MIN_SPEC = 0.00015  # stop when specular ratio < this

## 2. Load SPOTnet Model

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, 1, 1, bias=False), nn.BatchNorm2d(cout), nn.SiLU(inplace=True),
            nn.Conv2d(cout, cout, 3, 1, 1, bias=False), nn.BatchNorm2d(cout), nn.SiLU(inplace=True))
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        self.e1 = ConvBlock(3, base)
        self.e2 = ConvBlock(base, base*2)
        self.e3 = ConvBlock(base*2, base*4)
        self.b  = ConvBlock(base*4, base*4)
        self.d2 = ConvBlock(base*4+base*2, base*2)
        self.d1 = ConvBlock(base*2+base, base)
        self.head = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        s1 = self.e1(x)
        s2 = self.e2(F.avg_pool2d(s1, 2))
        s3 = self.e3(F.avg_pool2d(s2, 2))
        b  = self.b(s3)
        x  = F.interpolate(b,  scale_factor=2, mode='bilinear', align_corners=False)
        x  = self.d2(torch.cat([x, s2], dim=1))
        x  = F.interpolate(x,  scale_factor=2, mode='bilinear', align_corners=False)
        x  = self.d1(torch.cat([x, s1], dim=1))
        return self.head(x)

def to_tensor(img):
    if not img.flags['C_CONTIGUOUS']: img = np.ascontiguousarray(img)
    return torch.from_numpy(img).float().permute(2,0,1) / 255.0

@torch.no_grad()
def tiled_inference(net, img_rgb, tile=384, overlap=96):
    net.eval()
    H, W = img_rgb.shape[:2]
    X = to_tensor(img_rgb).unsqueeze(0).to(DEVICE)
    stride = tile - overlap
    pad_h = (math.ceil((H-tile)/stride)*stride+tile-H) if H>tile else tile-H
    pad_w = (math.ceil((W-tile)/stride)*stride+tile-W) if W>tile else tile-W
    X = F.pad(X, (0,pad_w,0,pad_h), mode='reflect')
    Hp, Wp = X.shape[-2:]
    acc = torch.zeros((H,W), device=DEVICE)
    cnt = torch.zeros((H,W), device=DEVICE)
    for y in range(0, Hp-tile+1, stride):
        for x in range(0, Wp-tile+1, stride):
            pred = torch.sigmoid(net(X[:,:,y:y+tile,x:x+tile]))[0,0]
            ye, xe = min(y+tile,H), min(x+tile,W)
            acc[y:ye,x:xe] += pred[:ye-y,:xe-x]
            cnt[y:ye,x:xe] += 1
    return (acc/cnt.clamp(min=1)).cpu().numpy()

# Load model
net = UNet(BASE_CH).to(DEVICE).eval()
ck  = torch.load(CKPT_PATH, map_location='cpu')
net.load_state_dict(ck['net'])
print('SPOTnet loaded!')

# Load LaMa
lama = SimpleLama()
print('LaMa loaded!')

## 3. Iterative Inpainting Function

In [ ]:
def iterative_inpaint(img_rgb, net, lama, max_iter=5, thr=0.5, dil_k=15, min_ratio=0.00015):
    """
    Iteratively detect and inpaint specular reflections.
    Stops early when specular ratio drops below min_ratio.
    """
    result   = img_rgb.copy()
    H, W     = result.shape[:2]
    total_px = H * W
    kernel   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dil_k, dil_k))

    for i in range(max_iter):
        soft  = tiled_inference(net, result, TILE, OVERLAP)
        mask  = (soft > thr).astype(np.uint8) * 255
        ratio = mask.sum() / 255 / total_px
        print(f'  Iter {i+1}: specular ratio = {ratio:.5f}')

        if ratio < min_ratio:
            print(f'  ✓ Clean! Stopping at iter {i+1}')
            break

        mask_dil = cv2.dilate(mask, kernel)
        result   = np.array(lama(Image.fromarray(result), Image.fromarray(mask_dil)))

    return result

## 4. Visual Test on Single Image

In [ ]:
# Pick a test frame
frames = sorted(glob.glob(f'{FRAMES_DIR}/*.jpg') + glob.glob(f'{FRAMES_DIR}/*.png'))
img_path = frames[0]  # change index as needed
print(f'Testing on: {img_path}')

img_rgb = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
result  = iterative_inpaint(img_rgb, net, lama, MAX_ITER, BIN_THR, DIL_K, MIN_SPEC)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img_rgb); axes[0].set_title('Original')
axes[1].imshow(result);  axes[1].set_title('Specular-Free (SPOTnet + LaMa)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 5. Process All Frames

In [ ]:
for img_path in tqdm(frames, desc='Processing'):
    name    = os.path.basename(img_path)
    img_rgb = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    result  = iterative_inpaint(img_rgb, net, lama, MAX_ITER, BIN_THR, DIL_K, MIN_SPEC)
    cv2.imwrite(os.path.join(OUTPUT_DIR, name), cv2.cvtColor(result, cv2.COLOR_RGB2BGR))

print(f'Done! Results saved to {OUTPUT_DIR}/')